# 🔀 Módulo 14 — Workflows: Edges Condicionales

> **Objetivo:** Crear bifurcaciones en el grafo del workflow basadas en el contenido del mensaje.

## ¿Qué es un edge condicional?

Un edge (arista) entre dos ejecutores que **sólo se activa si una condición se cumple**.

```
                       ┌─→ [PositiveHandler]   (si is_positive == True)
[SentimentAnalyzer] ───┤
                       └─→ [NegativeHandler]   (si is_positive == False)
```

## API

```python
.add_edge(from_node, to_node, condition=lambda msg: ...)
```

La función `condition` recibe el mensaje producido por `from_node`. Si devuelve `True`,
el edge se activa y el mensaje llega a `to_node`.

> 🎉 **No requiere Azure OpenAI** — los ejemplos son determinísticos.


In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from dataclasses import dataclass
from typing_extensions import Never
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, executor, handler

print("✅ Listo")


## 1️⃣ Edges directos en cadena (sin condición)

Para repaso: `add_edge(A, B)` sin `condition` crea un edge **incondicional**.


In [ ]:
@executor(id="uppercase")
async def uppercase(text: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(text.upper())


@executor(id="formatter")
async def formatter(text: str, ctx: WorkflowContext[str]) -> None:
    await ctx.send_message(f"[{text}]")


@executor(id="final_output")
async def final_output(text: str, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"Final: {text}")


workflow = (
    WorkflowBuilder(start_executor=uppercase)
    .add_edge(uppercase, formatter)
    .add_edge(formatter, final_output)
    .build()
)

events = await workflow.run("test data")
print(f"✅ Cadena directa → {events.get_outputs()}")


## 2️⃣ Edge condicional — bifurcación según sentimiento

El analyzer produce un `SentimentResult`. Dos edges con condiciones opuestas enrutan al handler correcto.

> Las condiciones son funciones puras `lambda msg: bool` — no requieren AI, ejecutables localmente.


In [ ]:
from dataclasses import dataclass


@dataclass
class SentimentResult:
    is_positive: bool
    text: str


_POSITIVE_WORDS = {"good", "great", "excellent", "happy", "love", "wonderful", "amazing"}


class SentimentAnalyzerExecutor(Executor):
    def __init__(self, id: str = "sentiment_analyzer"):
        super().__init__(id=id)

    @handler
    async def analyze(self, text: str, ctx: WorkflowContext[SentimentResult]) -> None:
        is_positive = any(w in text.lower() for w in _POSITIVE_WORDS)
        await ctx.send_message(SentimentResult(is_positive=is_positive, text=text))


@executor(id="positive_handler")
async def positive_handler(msg: SentimentResult, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"POSITIVE: {msg.text}")


@executor(id="negative_handler")
async def negative_handler(msg: SentimentResult, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"NEGATIVE: {msg.text}")


# --- Workflow positivo ---
analyzer = SentimentAnalyzerExecutor()

workflow_pos = (
    WorkflowBuilder(start_executor=analyzer)
    .add_edge(
        analyzer,
        positive_handler,
        condition=lambda msg: isinstance(msg, SentimentResult) and msg.is_positive,
    )
    .add_edge(
        analyzer,
        negative_handler,
        condition=lambda msg: isinstance(msg, SentimentResult) and not msg.is_positive,
    )
    .build()
)

events = await workflow_pos.run("This is a great day!")
print(f"✅ Input positivo  → {events.get_outputs()}")


## 3️⃣ Mismo workflow, input negativo → el otro camino

Verificamos que el edge condicional opuesto se active correctamente.


In [ ]:
# Reconstruimos el workflow con instancias frescas para el segundo run
analyzer2 = SentimentAnalyzerExecutor(id="sentiment_analyzer_2")

workflow_neg = (
    WorkflowBuilder(start_executor=analyzer2)
    .add_edge(
        analyzer2,
        positive_handler,
        condition=lambda msg: isinstance(msg, SentimentResult) and msg.is_positive,
    )
    .add_edge(
        analyzer2,
        negative_handler,
        condition=lambda msg: isinstance(msg, SentimentResult) and not msg.is_positive,
    )
    .build()
)

events = await workflow_neg.run("This is terrible and awful")
print(f"✅ Input negativo  → {events.get_outputs()}")


## 4️⃣ Switch-Case Edge Group — enrutamiento multi-rama

Cuando necesitas **más de dos caminos**, `add_switch_case_edge_group` es más limpio que
múltiples `add_edge(..., condition=...)`. Funciona como un `switch` / `match`:

```
                    ┌─→ [HighHandler]     (priority == "high")
[PriorityRouter] ──┤─→ [MediumHandler]   (priority == "medium")
                    └─→ [DefaultHandler]  (default / anything else)
```

Usa `Case(condition=..., target=...)` para cada rama explícita y `Default(target=...)` como
catch-all que garantiza que ningún mensaje se pierda.

> 🎉 **No requiere Azure OpenAI** — ejemplo 100% determinístico.

In [ ]:
from agent_framework import Case, Default


@dataclass
class PriorityMessage:
    priority: str  # "high", "medium", "low"
    text: str


class PriorityRouterExecutor(Executor):
    """Clasifica la prioridad del mensaje según palabras clave."""

    def __init__(self):
        super().__init__(id="priority_router")

    @handler
    async def route(self, text: str, ctx: WorkflowContext[PriorityMessage]) -> None:
        lower = text.lower()
        if any(w in lower for w in ("urgent", "critical", "asap", "urgente")):
            priority = "high"
        elif any(w in lower for w in ("important", "soon", "importante")):
            priority = "medium"
        else:
            priority = "low"
        await ctx.send_message(PriorityMessage(priority=priority, text=text))


@executor(id="high_handler")
async def high_handler(msg: PriorityMessage, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"🔴 HIGH: {msg.text}")


@executor(id="medium_handler")
async def medium_handler(msg: PriorityMessage, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"🟡 MEDIUM: {msg.text}")


@executor(id="default_handler")
async def default_handler(msg: PriorityMessage, ctx: WorkflowContext[Never, str]) -> None:
    await ctx.yield_output(f"🟢 LOW (default): {msg.text}")


# --- Workflow con switch-case ---
router = PriorityRouterExecutor()

workflow_switch = (
    WorkflowBuilder(start_executor=router)
    .add_switch_case_edge_group(
        router,
        [
            Case(condition=lambda m: isinstance(m, PriorityMessage) and m.priority == "high", target=high_handler),
            Case(condition=lambda m: isinstance(m, PriorityMessage) and m.priority == "medium", target=medium_handler),
            Default(target=default_handler),  # Catch-all para "low" o cualquier otro valor
        ],
    )
    .build()
)

# Probamos los 3 caminos
for msg in ["URGENT: server is down!", "Important meeting tomorrow", "Just a casual update"]:
    events = await workflow_switch.run(msg)
    print(f"  Input: {msg!r:45s} → {events.get_outputs()}")

print("\n✅ Switch-case completado — 3 ramas probadas")

## 5️⃣ Multi-Selection Edge Group — fan-out inteligente

A diferencia de switch-case (1 → **exactamente 1**), multi-selection permite enviar un mensaje
a **uno o más** targets simultáneamente. La `selection_func` recibe el mensaje y la lista de IDs
de los targets, y devuelve cuáles activar.

```
                         ┌─→ [Logger]       ← siempre
[ContentAnalyzer] ──────┤─→ [Notifier]     ← si contiene "alert"
                         └─→ [Archiver]     ← si texto > 50 chars
```

Ideal para pipelines donde diferentes ramas se activan según características del contenido.

> 🎉 **No requiere Azure OpenAI** — ejemplo 100% determinístico.

In [ ]:
@dataclass
class ContentMessage:
    text: str
    has_alert: bool
    is_long: bool


class ContentAnalyzerExecutor(Executor):
    """Analiza el contenido y marca propiedades para el routing."""

    def __init__(self):
        super().__init__(id="content_analyzer")

    @handler
    async def analyze(self, text: str, ctx: WorkflowContext[ContentMessage]) -> None:
        await ctx.send_message(ContentMessage(
            text=text,
            has_alert="alert" in text.lower(),
            is_long=len(text) > 50,
        ))


@executor(id="logger")
async def logger(msg: ContentMessage, ctx: WorkflowContext[Never, str]) -> None:
    """Siempre se ejecuta — registra todo."""
    await ctx.yield_output(f"📝 LOG: {msg.text}")


@executor(id="notifier")
async def notifier(msg: ContentMessage, ctx: WorkflowContext[Never, str]) -> None:
    """Solo se activa si hay una alerta."""
    await ctx.yield_output(f"🚨 ALERT NOTIFICATION: {msg.text}")


@executor(id="archiver")
async def archiver(msg: ContentMessage, ctx: WorkflowContext[Never, str]) -> None:
    """Solo se activa para mensajes largos."""
    await ctx.yield_output(f"🗄️ ARCHIVED ({len(msg.text)} chars): {msg.text[:30]}...")


def select_targets(msg: ContentMessage, target_ids: list[str]) -> list[str]:
    """Decide qué targets activar basándose en el contenido del mensaje."""
    # target_ids orden: [logger, notifier, archiver]
    logger_id, notifier_id, archiver_id = target_ids

    targets = [logger_id]  # Logger siempre activo

    if msg.has_alert:
        targets.append(notifier_id)

    if msg.is_long:
        targets.append(archiver_id)

    return targets


# --- Workflow con multi-selection ---
content_analyzer = ContentAnalyzerExecutor()

workflow_multi = (
    WorkflowBuilder(start_executor=content_analyzer)
    .add_multi_selection_edge_group(
        content_analyzer,
        [logger, notifier, archiver],
        selection_func=select_targets,
    )
    .build()
)

# Caso 1: Mensaje corto sin alerta → solo logger
print("─── Caso 1: Mensaje corto sin alerta ───")
events = await workflow_multi.run("Hello world")
for o in events.get_outputs():
    print(f"  {o}")

# Caso 2: Mensaje corto CON alerta → logger + notifier
print("\n─── Caso 2: Mensaje con alerta ───")
content_analyzer_2 = ContentAnalyzerExecutor()
content_analyzer_2._id = "content_analyzer_2"
wf2 = (
    WorkflowBuilder(start_executor=content_analyzer_2)
    .add_multi_selection_edge_group(content_analyzer_2, [logger, notifier, archiver], selection_func=select_targets)
    .build()
)
events = await wf2.run("ALERT: disk usage critical")
for o in events.get_outputs():
    print(f"  {o}")

# Caso 3: Mensaje largo CON alerta → logger + notifier + archiver (los 3!)
print("\n─── Caso 3: Mensaje largo con alerta → todos los targets ───")
content_analyzer_3 = ContentAnalyzerExecutor()
content_analyzer_3._id = "content_analyzer_3"
wf3 = (
    WorkflowBuilder(start_executor=content_analyzer_3)
    .add_multi_selection_edge_group(content_analyzer_3, [logger, notifier, archiver], selection_func=select_targets)
    .build()
)
events = await wf3.run("ALERT: The database connection pool has been exhausted and requests are timing out across all regions")
for o in events.get_outputs():
    print(f"  {o}")

print("\n✅ Multi-selection completado — fan-out dinámico según contenido")

## 🎯 Resumen

| Concepto | API | Cuándo usar |
|----------|-----|-------------|
| Edge directo | `.add_edge(A, B)` | Conexión 1→1 incondicional |
| Edge condicional | `.add_edge(A, B, condition=lambda msg: ...)` | Bifurcación binaria (if/else) |
| Switch-Case | `.add_switch_case_edge_group(A, [Case(...), Default(...)])` | Multi-rama exclusiva (1→exactamente 1) |
| Multi-Selection | `.add_multi_selection_edge_group(A, [...], selection_func=fn)` | Fan-out dinámico (1→N simultáneos) |
| Fan-in | `.add_fan_in_edge([A, B, C], D)` | Agregación de múltiples fuentes |

### Comparación rápida

```
Condicional:    A ──condition──→ B  (o no)
Switch-Case:    A ──→ B₁ XOR B₂ XOR B₃   (exactamente uno)
Multi-Selection: A ──→ B₁ AND/OR B₂ AND/OR B₃  (uno o más)
```

> 💡 Las condiciones son **funciones Python normales** — puedes usar cualquier lógica
> (incluyendo llamadas síncronas a APIs, comprobaciones de tipos, etc.).

**Siguiente módulo →** [Módulo 15 — Workflows: Eventos y Loops](./15_workflows_events.ipynb)